# AutoML Tabular Workflow Pipelines with Modern Python Dependencies

This notebook demonstrates training classification models on Google Cloud Vertex AI using AutoML Tabular Workflows.
It updates official Google Cloud sample notebooks to modern Python package versions and APIs using `google-cloud-pipeline-components>=2.22.0` (`google_cloud_pipeline_components.v1.automl.tabular`).

### Objectives
1. Configure and launch a customized AutoML Tabular training pipeline on the Bank Marketing dataset.
2. Extract the hyperparameter tuning result artifact from Stage 1.
3. Launch a Skip Architecture Search AutoML Tabular pipeline reusing Stage 1 tuning results to save time and cost.

In [1]:
from dotenv import load_dotenv
from google.cloud import aiplatform

from tabflows import (
    TabularPipelineConfig,
    build_automl_tabular_pipeline,
    build_skip_architecture_search_pipeline,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment and libraries loaded successfully.")

Environment and libraries loaded successfully.


In [2]:
# TabularPipelineConfig automatically loads GCP_PROJECT, GCP_LOCATION, and GCP_BUCKET_URI from .env
config = TabularPipelineConfig()

print(f"Project ID: {config.project_id}")
print(f"Location: {config.location}")
print(f"Bucket URI: {config.bucket_uri}")
print(f"Pipeline Root DIR: {config.root_dir}")
print(f"Transform Config Path: {config.transform_config_path}")

Project ID: hybrid-vertex
Location: us-central1
Bucket URI: gs://jts-tabflows-v1
Pipeline Root DIR: gs://jts-tabflows-v1/automl_tabular_pipeline
Transform Config Path: gs://jts-tabflows-v1/automl_tabular_pipeline/transform_config_unique.json


In [3]:
# Build pipeline template and parameter dictionary
template_path, parameter_values = build_automl_tabular_pipeline(config)

job_id = "automl-tabular-run-01"
aiplatform.init(project=config.project_id, location=config.location)

job = aiplatform.PipelineJob(
    display_name=job_id,
    location=config.location,
    template_path=template_path,
    job_id=job_id,
    pipeline_root=config.root_dir,
    parameter_values=parameter_values,
    enable_caching=False,
)

# To execute job on Vertex AI: job.run()
print(f"PipelineJob object created successfully: {job_id}")

PipelineJob object created successfully: automl-tabular-run-01


## Skip Architecture Search Pipeline

Reusing the hyperparameter tuning result from the stage-1 tuner task reduces training time and cost.

Extract the tuning result artifact URI from `automl-tabular-stage-1-tuner` and pass it to `build_skip_architecture_search_pipeline`.

In [4]:
# Example: Extract tuning result artifact URI after job execution:
# pipeline_task_details = job.gca_resource.job_detail.task_details
# stage_1_task = get_task_detail(pipeline_task_details, "automl-tabular-stage-1-tuner")
# stage_1_tuning_result_artifact_uri = (
#     stage_1_task.outputs["tuning_result_output"].artifacts[0].uri
# )

stage_1_tuning_result_artifact_uri = f"{config.root_dir}/tuning_result_artifact"

skip_template_path, skip_parameter_values = build_skip_architecture_search_pipeline(
    config=config,
    stage_1_tuning_result_artifact_uri=stage_1_tuning_result_artifact_uri,
)

skip_job_id = "automl-tabular-skip-search-01"
skip_job = aiplatform.PipelineJob(
    display_name=skip_job_id,
    location=config.location,
    template_path=skip_template_path,
    job_id=skip_job_id,
    pipeline_root=config.root_dir,
    parameter_values=skip_parameter_values,
    enable_caching=False,
)

# To execute job on Vertex AI: skip_job.run()
print(f"Skip Architecture Search PipelineJob object created successfully: {skip_job_id}")

Skip Architecture Search PipelineJob object created successfully: automl-tabular-skip-search-01
